In [ ]:
"""
fitters_test.ipynb

Tests the fitter object, meant to 
fit all models to a given session.

Author: Stellina X. Ao
Created: 2026-03-05
Last Modified: 2026-03-10
Python Version: 3.11.14
"""

from sg.fitter import LVMFamily
from src.squiggs.neuron_viewer import NeuronViewer
from src.squiggs.renderers import FitRenderer
from sg.eval_models import plot_summary

import scienceplots  # noqa: F401
import shutup
import matplotlib.pyplot as plt
import pickle
import numpy as np
from pathlib import Path

%load_ext autoreload
%autoreload 2

# pretty plots
plt.style.use(["nature"])
plt.rcParams["figure.dpi"] = 200
%matplotlib widget
%config InlineBackend.print_figure_kwargs = {'bbox_inches':None}

# seed and suppress warnings :-)
# fitlvm_utils.seed()
shutup.please()

In [ ]:
"""
TODO:
- does adding block improve r2?
- look at fits of neurons with and without block
- what about adding ReLU to the taskvar model and removing block?
- add save feature to renderer

"""

## Fit one session with 4 additive latents

In [ ]:
trial_data_all = np.load(
    "../vars/trial_data_all_MM012_MM013_all5.npz", allow_pickle=True
)["arr_0"]
session_data_all = np.load(
    "../vars/session_data_all_MM012_MM013_all5.npz", allow_pickle=True
)["arr_0"]
unit_spike_times_all = np.load(
    "../vars/unit_spike_times_all_MM012_MM013_all5.npz", allow_pickle=True
)["arr_0"]
regions_all = np.load("../vars/regions_all_MM012_MM013_all5.npz", allow_pickle=True)[
    "arr_0"
]

subj_idx = 0
sess_idx = 5

unit_spike_times = unit_spike_times_all[subj_idx][sess_idx]
trial_data = trial_data_all[subj_idx][sess_idx]
session_data = session_data_all[subj_idx][sess_idx]
regions = regions_all[subj_idx][sess_idx]
trial_data["block_side"] = np.where(trial_data["block_side"] == "left", 1, -1)

In [ ]:
family = LVMFamily(
    trial_data=trial_data,
    spike_times=unit_spike_times,
    session_data=session_data,
    regions=regions,
    n_latents_mult=4,
    n_latents_addt=4,
    sanity_check=0,
    task_vars=["response", "rewarded", "block_side", "response_prev", "rewarded_prev"],
)
family.fit_all()

In [ ]:
family.eval()
family.res_taskvar[
    "r2test"
].mean()  # 0.3024 for only four taskvars, 0.3089 for block added

In [ ]:
render_baseline = FitRenderer(
    family.mod_taskvar,  # TODO: mod_ae_gain looks so weird
    x=family.test_dl.dataset[:],
    y=family.test_dl.dataset[:]["robs"].detach().cpu(),
    save_subdir=Path("model_fits") / "debug_logs" / "block",
)

nv = NeuronViewer(num_units=render_baseline.y.shape[1], render_func=render_baseline)

In [ ]:
from sg.eval_models import res_taskvar_corr

res_taskvar_corr(family, plot_r2_dist=False)

In [ ]:
family.eval()
family.res_gain["r2test"].mean()

In [ ]:
potato = (
    2 * np.sin(np.arange(family.psths["DMS"].shape[1]))
    + 0.5 * np.cos(np.arange(family.psths["DMS"].shape[1]))
    + 2.5
)

In [ ]:
plot_summary(family, model=family.mod_gain, potato=potato, mode="gain")

In [ ]:
# TODO:
# probably happened as i migrated to oop, check in the notebook that these residuals are not overshooting
# yeah fixable, but tomorrow problem
# priority #1 is to figure out what is wrong with the encoder, why is is just overshooting?
# there has to be something wrong with the bias (i remember saying that i unfroze it at some point)
# check that it's not block
# dfs? yeah, do i really need train test and val? <-- most likely <== nvm, didn't do anything noticeable
# or alternatively, adding block? i doubt it..

In [ ]:
# with open("../vars/family.pkl", "wb") as f:
# pickle.dump(family, f)
# with open("../vars/family.pkl", "rb") as f:
#     family = pickle.load(f)

In [ ]:
render_baseline = FitRenderer(
    family.mod_ae_gain,
    x=family.test_dl.dataset[:],
    y=family.test_dl.dataset[:]["robs"].detach().cpu(),
)
nv = NeuronViewer(num_units=render_baseline.y.shape[1], render_func=render_baseline)

In [ ]:
plot_summary(family, family.mod_offset, potato=family.strategy, mode="offset")

## Plot r2 matrix

In [ ]:
import numpy as np

m_latents = np.arange(1, 10 + 1)  # 10, 10)
a_latents = np.arange(1, 10 + 1)  # 10, 10)

r2s = np.zeros((len(m_latents), len(a_latents)))
for i, m in enumerate(m_latents):
    for j, a in enumerate(a_latents):
        with open(f"../vars/families/family-m{int(m)}a{int(a)}.pkl", "rb") as f:
            family_ = pickle.load(f)
            family_.eval()
            r2s[i, j] = family_.res_affine["r2test"].mean()

In [ ]:
import matplotlib.pyplot as plt

plt.style.use(["nature"])
plt.rcParams["figure.dpi"] = 200
%matplotlib widget

plt.figure()
plt.imshow(r2s, origin="lower", interpolation=None)
plt.xlabel("N. Additive Latents")
plt.ylabel("N. Multiplicative Latents")
plt.xticks(np.arange(len(a_latents)), np.arange(1, len(a_latents) + 1))
plt.yticks(np.arange(len(a_latents)), np.arange(1, len(a_latents) + 1))
plt.colorbar()
plt.tight_layout()
plt.show()

## Check latents

In [ ]:
plt.figure()
fig, axes = plt.subplots(ncols=2)
axes[0].imshow(family.train_dl.dataset[:]["dfs"].T)
axes[1].imshow(family.test_dl.dataset[:]["dfs"].T)
plt.show()

In [ ]:
m = 4
a = 10

with open(f"../vars/families/midnight-run/family-m{m}a{a}.pkl", "rb") as f:
    family = pickle.load(f)
family.eval()

plot_summary(family, family.mod_offset, potato=family.strategy, mode="offset")

## Correlate latents

In [ ]:
# m=4
# a=10
# with open(f"../vars/families/midnight-run/family-m{m}a{a}.pkl", "rb") as f:
#     family = pickle.load(f)

# # only focus on offset for now
# model =